In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD CONFIG FROM NOTEBOOKS 05, 40, 52, 55, 56
# =============================================================================
import os
import sys
import json
import time
import warnings
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Config From Notebooks 05, 40, 52, 55, 56")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"
NB05_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_05_summary.json"
NB40_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_40_summary.json"
NB50_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_50_summary.json"
NB52_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_52_summary.json"
NB55_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_55_summary.json"
NB56_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_56_summary.json"

for _p, _fix in [
    (CONFIG_PATH, "run 01_business_understanding.ipynb first"),
    (NB05_SUMMARY_PATH, "run 05_model_development.ipynb first"),
    (NB40_SUMMARY_PATH, "run 40_dynamic_behavioral_scoring_validation_deployment.ipynb first (Problem 6)"),
    (NB50_SUMMARY_PATH, "run 50_collections_optimization_business_understanding.ipynb first (Problem 9)"),
    (NB52_SUMMARY_PATH, "run 52_collections_optimization_validation_deployment.ipynb first (Problem 9) -- "
                         "Problem 12 reuses Problem 9's real, validated propensity-to-cure model and "
                         "treatment-tier policy, never a fresh fit"),
    (NB55_SUMMARY_PATH, "run 55_credit_line_management_modeling.ipynb first (Problem 10) -- Problem 12 reuses "
                         "Problem 10's real, scored customer worklist directly rather than re-deriving it"),
    (NB56_SUMMARY_PATH, "run 56_credit_line_management_validation_deployment.ipynb first (Problem 10)"),
]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} not found.\nFix: {_fix}")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)
with open(NB05_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB05_SUMMARY = json.load(f)
with open(NB40_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB40_SUMMARY = json.load(f)
with open(NB50_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB50_SUMMARY = json.load(f)
with open(NB52_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB52_SUMMARY = json.load(f)
with open(NB55_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB55_SUMMARY = json.load(f)
with open(NB56_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB56_SUMMARY = json.load(f)

# --- Four real, already-validated sources Problem 12 aggregates -- read via
#     each producing notebook's OWN recorded path, never re-derived or
#     guessed (the canonical-source-of-truth pattern this platform adopted
#     after a real FileNotFoundError was found and fixed in Notebook 50). ---
CHAMPION_NAME = NB05_SUMMARY["champion_model"]
CHAMPION_METRICS = NB05_SUMMARY["champion_metrics"]
STATIC_PD_AUC = CHAMPION_METRICS.get("holdout_auc")

P6_DEPLOYMENT_POLICY_PATH = Path(NB40_SUMMARY["deployment_policy_path"]) if "deployment_policy_path" in NB40_SUMMARY \
    else Path(NB40_SUMMARY.get("policy_path", ""))
P6_MODEL_PATH = Path(NB40_SUMMARY["model_path"])
P6_PREPROCESSING_PATH = Path(NB40_SUMMARY["preprocessing_path"])
P6_WINNING_W = NB40_SUMMARY["winning_w"]
P6_RECOMMENDED_FOR_PRODUCTION = NB40_SUMMARY["recommended_for_production"]

P9_MODEL_PATH = Path(NB52_SUMMARY["model_path"])
P9_DEPLOYMENT_POLICY_PATH = Path(NB52_SUMMARY["deployment_policy_path"])
with open(P9_DEPLOYMENT_POLICY_PATH, "r", encoding="utf-8") as f:
    P9_DEPLOYMENT_POLICY = json.load(f)
P9_COLLECTIONS_ELIGIBLE_STATES = P9_DEPLOYMENT_POLICY["collections_eligible_states"]
P9_TREATMENT_TIERS = P9_DEPLOYMENT_POLICY["treatment_tier_policy"]["tiers"]
P9_RECOMMENDED_FOR_PRODUCTION = NB52_SUMMARY["recommended_for_production"]
P9_CURE_LABEL_COVERAGE_PCT = NB50_SUMMARY["cure_label_upper_bound_coverage_pct"]

P10_WORKLIST_PATH = Path(NB55_SUMMARY["worklist_path"])
P10_DEPLOYMENT_POLICY_PATH = Path(NB56_SUMMARY["deployment_policy_path"])
with open(P10_DEPLOYMENT_POLICY_PATH, "r", encoding="utf-8") as f:
    P10_DEPLOYMENT_POLICY = json.load(f)
P10_RISK_LEVEL_NAMES = P10_DEPLOYMENT_POLICY["risk_level_names"]
P10_TREND_NAMES = P10_DEPLOYMENT_POLICY["trend_names"]
P10_ACTION_TIER_MATRIX = P10_DEPLOYMENT_POLICY["action_tier_matrix"]
P10_RECOMMENDED_FOR_PRODUCTION = NB56_SUMMARY["recommended_for_production"]

for _flag, _label in [
    (P6_RECOMMENDED_FOR_PRODUCTION, "Problem 6 (dynamic behavioral PD)"),
    (P9_RECOMMENDED_FOR_PRODUCTION, "Problem 9 (collections propensity-to-cure)"),
    (P10_RECOMMENDED_FOR_PRODUCTION, "Problem 10 (credit-line risk-level/trend)"),
]:
    if not _flag:
        print(
            f"WARNING: {_label} is NOT currently recommended for production. Problem 12 still reuses its "
            "real, measured score/output verbatim (the signal itself is real regardless of the recommendation "
            "flag), but this is noted honestly here and carried into the unified profile's own KPI validation "
            "in Notebook 63, rather than silently assumed away."
        )

for _p, _label in [
    (P6_MODEL_PATH, "Problem 6's persisted model"), (P6_PREPROCESSING_PATH, "Problem 6's preprocessing artifacts"),
    (P9_MODEL_PATH, "Problem 9's persisted propensity model"),
    (P10_WORKLIST_PATH, "Problem 10's real scored worklist"),
]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} not found ({_label}).\nFix: re-run the notebook that produces it.")

PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}
DETECTED_LOGICAL_CORES = PROJECT_CONFIG["hardware"]["logical_cores_detected"]
DETECTED_TOTAL_RAM_BYTES = PROJECT_CONFIG["resource_limits"]["total_ram_bytes_detected"]
RANDOM_SEED = PROJECT_CONFIG["random_seed"]

if "customer_intelligence_policy" in PILLAR_DIRS:
    CUSTOMER_INTELLIGENCE_POLICY_DIR = PILLAR_DIRS["customer_intelligence_policy"]
else:
    CUSTOMER_INTELLIGENCE_POLICY_DIR = (
        PROJECT_ROOT / "Phase5_Customer_Business_Intelligence"
        / "Problem12_360_Customer_Intelligence" / "policy"
    )
    print(f"NOTE: 'customer_intelligence_policy' not in pillar_dirs -- using fallback: "
          f"{CUSTOMER_INTELLIGENCE_POLICY_DIR}")
CUSTOMER_INTELLIGENCE_POLICY_DIR.mkdir(parents=True, exist_ok=True)

print(f"Loaded config from                            : {CONFIG_PATH}")
print(f"Champion architecture (Problem 1, measured)   : {CHAMPION_NAME}, holdout AUC {STATIC_PD_AUC}")
print(f"Problem 6 dynamic PD (measured)               : W={P6_WINNING_W}, "
      f"recommended={P6_RECOMMENDED_FOR_PRODUCTION}")
print(f"Problem 9 propensity-to-cure (measured)       : recommended={P9_RECOMMENDED_FOR_PRODUCTION}, "
      f"{len(P9_TREATMENT_TIERS)} treatment tiers, eligible states={P9_COLLECTIONS_ELIGIBLE_STATES}")
print(f"Problem 10 risk-level/trend (measured)        : recommended={P10_RECOMMENDED_FOR_PRODUCTION}, "
      f"{len(P10_ACTION_TIER_MATRIX)}-cell action matrix")
print(f"Policy artifacts will be written under: {CUSTOMER_INTELLIGENCE_POLICY_DIR}")
print("\n✅ Section 1 complete.")


# =============================================================================
# SECTION 2: WARP HARDWARE CONFIGURATION & LIBRARY IMPORTS (PHASE 4 CAP,
#            92% CPU / 92% RAM, CARRIED FORWARD -- NO NEW INCIDENT HAS
#            OCCURRED TO WARRANT CHANGING IT FOR PHASE 5)
# =============================================================================
_section("SECTION 2: WARP Hardware Configuration & Library Imports")

_PHASE4_CPU_FRACTION_CAP = 0.92
_PHASE4_RAM_FRACTION_CAP = 0.92
_historical_thread_count = PROJECT_CONFIG["resource_limits"]["warp_thread_count"]
_historical_max_ram_bytes = PROJECT_CONFIG["resource_limits"]["max_ram_bytes"]
WARP_THREAD_COUNT = min(
    _historical_thread_count,
    max(1, round(DETECTED_LOGICAL_CORES * _PHASE4_CPU_FRACTION_CAP)),
)
MAX_RAM_BYTES = min(
    _historical_max_ram_bytes,
    round(DETECTED_TOTAL_RAM_BYTES * _PHASE4_RAM_FRACTION_CAP),
)

os.environ["POLARS_MAX_THREADS"] = str(WARP_THREAD_COUNT)
warnings.filterwarnings("ignore", category=UserWarning)

import logging
logger = logging.getLogger("amex_platform")
logger.setLevel(logging.INFO)
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)

missing = []
try:
    import polars as pl
except ImportError:
    missing.append("polars")
try:
    import psutil
except ImportError:
    missing.append("psutil")
if missing:
    raise ImportError(
        "Missing required package(s): " + ", ".join(missing) + "\n"
        "Fix: run this in a terminal, then re-run this cell:\n"
        f"    pip install {' '.join(missing)}"
    )


def _rss_gb() -> float:
    return psutil.Process().memory_info().rss / 1e9


def _available_ram_gb() -> float:
    return psutil.virtual_memory().available / 1e9


# --- Two-tier (warn/hard-fail) RAM pre-flight guard, established in
#     Notebooks 51/52 after a real 45-minute freeze the user hit -- reused
#     verbatim, not a single blind cutoff. ---
_available_ram_gb_at_start = _available_ram_gb()
_comfortable_available_ram_gb = 0.50 * (MAX_RAM_BYTES / 1e9)
_min_required_available_ram_gb = 0.25 * (MAX_RAM_BYTES / 1e9)
if _available_ram_gb_at_start < _min_required_available_ram_gb:
    raise RuntimeError(
        f"Only {_available_ram_gb_at_start:.2f} GB of system RAM is available, which is below the "
        f"{_min_required_available_ram_gb:.2f} GB floor this notebook needs. Close other Jupyter kernels / "
        f"memory-heavy applications, confirm available RAM with `psutil.virtual_memory().available / 1e9` in "
        f"a fresh cell, then re-run this notebook from the top."
    )
if _available_ram_gb_at_start < _comfortable_available_ram_gb:
    print(f"⚠️  WARNING: only {_available_ram_gb_at_start:.2f} GB of system RAM is available "
          f"(comfortable margin is {_comfortable_available_ram_gb:.2f} GB). Proceeding.")
else:
    print(f"RAM pre-flight check passed: {_available_ram_gb_at_start:.2f} GB available >= "
          f"{_comfortable_available_ram_gb:.2f} GB comfortable margin.")

logger.info(f"Polars thread pool configured to {WARP_THREAD_COUNT}/{DETECTED_LOGICAL_CORES} threads")
print(f"Process RSS at Section 2 start: {_rss_gb():.2f} GB")
print(f"Configured RAM ceiling: {MAX_RAM_BYTES / 1e9:.1f} GB")
print("\n✅ Section 2 complete.")


# =============================================================================
# SECTION 3: RESOLVE REAL DATA PATHS
# =============================================================================
_section("SECTION 3: Resolve Real Data Paths")

_raw_candidates = []
if "raw_data_dir" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["raw_data_dir"]) / "train_data.csv")
if "data_root" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["data_root"]) / "train_data.csv")
_raw_candidates.append(PROJECT_ROOT.parent / "Raw Data From Kaggle" / "train_data.csv")

RAW_TRAIN_DATA_PATH = None
for _candidate in _raw_candidates:
    if _candidate.exists() and _candidate.stat().st_size > 1_000_000:
        RAW_TRAIN_DATA_PATH = _candidate
        break
if RAW_TRAIN_DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find the raw train_data.csv. Checked:\n" + "\n".join(f"  - {c}" for c in _raw_candidates)
    )
RAW_TRAIN_LABELS_PATH = RAW_TRAIN_DATA_PATH.parent / "train_labels.csv"
if not RAW_TRAIN_LABELS_PATH.exists():
    raise FileNotFoundError(f"{RAW_TRAIN_LABELS_PATH} not found.")

print(f"Raw train_data.csv  : {RAW_TRAIN_DATA_PATH}")
print(f"Raw train_labels.csv: {RAW_TRAIN_LABELS_PATH}")
print("\n✅ Section 3 complete.")


# =============================================================================
# SECTION 4: REAL PER-CUSTOMER STATEMENT-COUNT DISTRIBUTION (REUSED MEASURE)
# =============================================================================
_section("SECTION 4: Real Per-Customer Statement-Count Distribution")

print("Reading real per-statement (raw, pre-aggregation) data from: " + str(RAW_TRAIN_DATA_PATH))
_t0 = time.time()
_statement_counts = (
    pl.scan_csv(RAW_TRAIN_DATA_PATH)
    .select(pl.col("customer_ID"))
    .group_by("customer_ID")
    .agg(pl.len().alias("n_statements"))
    .collect()
)
print(f"Grouped in {time.time() - _t0:.1f}s. Process RSS: {_rss_gb():.2f} GB, "
      f"available RAM: {_available_ram_gb():.2f} GB")
_n_customers = _statement_counts.height
_counts_series = _statement_counts["n_statements"]
_n_dynamic_pd_eligible = int((_counts_series >= P6_WINNING_W).sum())
DYNAMIC_PD_ELIGIBILITY_COVERAGE_PCT = 100.0 * _n_dynamic_pd_eligible / _n_customers
print(f"Total real customers                         : {_n_customers:,}")
print(f"Dynamic-PD eligible (>= {P6_WINNING_W} statements, reused from Problem 6/10): "
      f"{_n_dynamic_pd_eligible:,} ({DYNAMIC_PD_ELIGIBILITY_COVERAGE_PCT:.1f}%)")
print(f"Collections-eligible coverage (Problem 9's own real, measured figure, reused verbatim): "
      f"~{P9_CURE_LABEL_COVERAGE_PCT:.1f}% (upper bound -- customers ever in a non-'Low Severity' state)")
print("\n✅ Section 4 complete.")


# =============================================================================
# SECTION 5: BUSINESS UNDERSTANDING -- 360° CUSTOMER INTELLIGENCE
# =============================================================================
_section("SECTION 5: Business Understanding -- 360 Degree Customer Intelligence")

print(
    "PROBLEM 12 -- 360 DEGREE CUSTOMER INTELLIGENCE (multi-signal customer profile aggregation)\n\n"
    "Business case: by this point the platform has built four independent, separately validated real views "
    "of customer risk -- Problem 1's static, whole-history PD; Problem 6's dynamic, monthly-refreshed PD; "
    "Problem 9's collections propensity-to-cure score (for customers already in a delinquent state); and "
    "Problem 10's risk-level/trend-based credit-line action. No single one of these is 'the' customer risk "
    "profile -- each answers a different operational question, and a risk officer or relationship manager "
    "reviewing one customer today has to open four separate systems to see all four. Problem 12 builds the "
    "real, single unified profile that composes all four real signals into one per-customer record, so every "
    "downstream consumer (Problem 13's profitability model, Problem 14's executive dashboard, and any real "
    "operational review) reads from one place instead of four.\n\n"
    "WHY THIS IS A GENUINELY NEW ARTIFACT, NOT A REPACKAGING OF PROBLEMS 1/6/9/10: none of the four upstream "
    "notebooks computes a combined view, and naively averaging or concatenating their outputs is not "
    "guaranteed to preserve -- let alone improve -- each signal's own real discriminatory power. Notebook 63 "
    "computes a real UNIFIED_RISK_SCORE as an explicit, documented composite of the four signals and then "
    "VALIDATES -- with a real hard-gate KPI, not an assumed relationship -- that this composite score is at "
    "least as predictive of real eventual default as the single best real input signal alone. If that gate "
    "fails, the honest conclusion is that naive aggregation destroys signal for this dataset, and that is "
    "reported plainly rather than hidden behind a plausible-looking composite number.\n\n"
    "COVERAGE HONESTY (same standing caveat as Problems 9/10): not every customer has all four real signals. "
    "Every customer with a real static PD (effectively the whole book) can get a real risk-level/trend "
    "read once they clear Problem 6's real statement-count floor -- but Problem 9's real propensity-to-cure "
    "score only exists for customers who have actually been in a non-'Low Severity' delinquency state at "
    "some point (Problem 9's own real, measured coverage, reused verbatim above), since a cure propensity is "
    "only a meaningful question for someone who has something to cure. This notebook's KPI targets (Section "
    "7) require every customer to carry an honestly NULL collections field when they are not collections-"
    "eligible -- never a fabricated propensity score for a customer who was never delinquent."
)
print("\n✅ Section 5 complete.")


# =============================================================================
# SECTION 6: UNIFIED PROFILE SCHEMA & COMPOSITE SCORE DEFINITION (ASSUMPTION)
# =============================================================================
_section("SECTION 6: Unified Profile Schema & Composite Score Definition (ASSUMPTION)")

# ASSUMPTION: the real per-customer fields the unified profile carries.
# STATIC_PD, DYNAMIC_PD, RISK_LEVEL/TREND/ACTION and PD_TREND are always
# real and present once a customer clears the dynamic-PD eligibility floor;
# COLLECTIONS_PROPENSITY/TREATMENT_TIER are honestly NULL outside Problem
# 9's own real eligible-states population -- never imputed.
UNIFIED_PROFILE_FIELDS = [
    "customer_ID", "STATIC_PD", "DYNAMIC_PD", "PD_TREND", "RISK_LEVEL", "TREND_SEGMENT",
    "CREDIT_LINE_ACTION", "COLLECTIONS_ELIGIBLE", "PROPENSITY_TO_CURE", "TREATMENT_TIER",
    "UNIFIED_RISK_SCORE", "UNIFIED_RISK_GRADE",
]

# ASSUMPTION: composite weighting. DYNAMIC_PD gets the largest weight (most
# current real read of behavior); STATIC_PD contributes the origination-time
# context; a real collections propensity signal (when present) shifts the
# composite toward higher risk for customers with low real cure propensity,
# scaled small since it only applies to the collections-eligible minority.
# Weights sum to 1.0 for the two always-present components; the collections
# adjustment is an additive real term applied only where real data exists.
UNIFIED_SCORE_WEIGHTS = {
    "static_pd_weight": 0.35,
    "dynamic_pd_weight": 0.65,
    "collections_adjustment_weight": 0.10,
    "description": (
        "ASSUMPTION -- DYNAMIC_PD weighted higher than STATIC_PD (0.65 vs 0.35) because it is the platform's "
        "most current real read of behavior (same precedence Problem 10 already gave it). Where a real "
        "collections propensity-to-cure score exists (Problem 9), UNIFIED_RISK_SCORE = "
        "0.35*STATIC_PD + 0.65*DYNAMIC_PD + 0.10*(1 - PROPENSITY_TO_CURE), renormalized to sum to 1.0 across "
        "the active terms; where it does not exist, UNIFIED_RISK_SCORE = 0.35*STATIC_PD + 0.65*DYNAMIC_PD "
        "unchanged. This is validated, not assumed correct -- see the composite_non_inferiority KPI below."
    ),
}

# ASSUMPTION: 3-tier grading of the composite score, reusing this platform's
# established tertile convention (Problems 4/8/10) for cross-platform
# continuity. Cut VALUES are fit fresh on the real scored population in
# Notebook 63 (the UNIFIED_RISK_SCORE distribution does not exist until all
# four real signals have actually been composed) -- only the convention is
# policy here, mirroring Problem 10's own precedent (Notebook 54 set the
# convention, Notebook 55 fit the real cut values).
UNIFIED_RISK_GRADE_NAMES = ["Low Risk", "Medium Risk", "High Risk"]
UNIFIED_RISK_GRADE_CUT_PERCENTILES = [33.333, 66.667]

print(f"UNIFIED_PROFILE_FIELDS ({len(UNIFIED_PROFILE_FIELDS)} real fields): {UNIFIED_PROFILE_FIELDS}")
print(f"UNIFIED_SCORE_WEIGHTS (ASSUMPTION): static={UNIFIED_SCORE_WEIGHTS['static_pd_weight']}, "
      f"dynamic={UNIFIED_SCORE_WEIGHTS['dynamic_pd_weight']}, "
      f"collections_adj={UNIFIED_SCORE_WEIGHTS['collections_adjustment_weight']}")
print(f"UNIFIED_RISK_GRADE_NAMES (ASSUMPTION, tertile convention): {UNIFIED_RISK_GRADE_NAMES}")
print("\n✅ Section 6 complete.")


# =============================================================================
# SECTION 7: KPI TARGETS -- PROFILE COMPLETENESS & COMPOSITE NON-INFERIORITY
# =============================================================================
_section("SECTION 7: KPI Targets -- Profile Completeness & Composite Non-Inferiority")

CUSTOMER_INTELLIGENCE_KPI_TARGETS = {
    "profile_completeness": {
        "description": (
            "PRIMARY, hard-gating KPI: every real customer who clears Problem 6's dynamic-PD eligibility "
            "floor must receive a real, non-null value for STATIC_PD, DYNAMIC_PD, PD_TREND, RISK_LEVEL, "
            "TREND_SEGMENT, CREDIT_LINE_ACTION, and UNIFIED_RISK_SCORE in Notebook 63's unified profile -- "
            "zero silently dropped or null rows for any of these seven always-applicable fields. "
            "COLLECTIONS_ELIGIBLE, PROPENSITY_TO_CURE, and TREATMENT_TIER are honestly allowed to be null "
            "for customers outside Problem 9's own real eligible-states population -- this is a real data "
            "limitation, not a completeness failure."
        ),
        "hard_gate": True,
    },
    "composite_non_inferiority": {
        "description": (
            "NEW hard-gating KPI for this problem (no Problem 4/6/8/9/10 analogue -- combining four real "
            "signals into one composite has no prior platform precedent to reuse): UNIFIED_RISK_SCORE's real "
            "holdout ROC-AUC against the real observed default label must be >= "
            "max(STATIC_PD's real AUC, DYNAMIC_PD's real AUC) minus a small ASSUMPTION tolerance of 0.005 "
            "(allows for real sampling noise while still requiring genuine non-inferiority, not just "
            "'close'). This is the specific, testable claim this whole problem depends on -- that "
            "aggregating four real signals into one composite does not destroy the signal any one of them "
            "already carried. If this gate fails, the honest conclusion is reported plainly: this "
            "notebook's composite weighting (Section 6) needs revision, not that the underlying signals "
            "are bad."
        ),
        "auc_tolerance": 0.005,
        "hard_gate": True,
    },
    "min_tier_population_pct": 10.0,
    "min_tier_population_pct_description": (
        "ASSUMPTION -- each of the 3 real UNIFIED_RISK_GRADE tiers must hold >= 10% of the eligible "
        "validation population, reusing this platform's Problem 4/8 single-axis threshold convention."
    ),
    "metrics_suite_requirement": (
        "STANDING RULE (user directive, carried from Problems 6-10): Notebook 63/64 must compute and "
        "DISPLAY -- inline in the notebook AND in this problem's Word/Excel/HTML reports -- the full "
        "classification metrics suite: ROC-AUC, PR-AUC, Accuracy, Precision, Recall, F1, Specificity, Log "
        "Loss, Matthews Correlation Coefficient, and a full confusion matrix, both for UNIFIED_RISK_SCORE "
        "alone and for each of its real input signals, so the non-inferiority claim is auditable."
    ),
    "elevated_reporting_requirement": (
        "STANDING RULE (user directive, carried from Problems 7-10): Problem 12's Word report must "
        "synthesize MAXIMUM DETAIL from every one of this problem's notebooks (62-65), with a narrative "
        "'story' paragraph below every chart. Problem 12's HTML report must be an advanced, 'global "
        "standard' interactive dashboard with slicers, filters, full legends, and interactive KPI cards -- "
        "built in Notebook 65, and must double as the real per-customer 360-degree profile viewer the "
        "master plan names as this problem's deliverable."
    ),
    "static_pd_reference_auc": STATIC_PD_AUC,
    "problem_6_reference": {"winning_w": P6_WINNING_W, "recommended_for_production": P6_RECOMMENDED_FOR_PRODUCTION},
    "problem_9_reference": {"recommended_for_production": P9_RECOMMENDED_FOR_PRODUCTION,
                             "cure_label_coverage_pct": P9_CURE_LABEL_COVERAGE_PCT},
    "problem_10_reference": {"recommended_for_production": P10_RECOMMENDED_FOR_PRODUCTION},
}
print(f"profile_completeness (hard gate): {CUSTOMER_INTELLIGENCE_KPI_TARGETS['profile_completeness']['hard_gate']}")
print(f"composite_non_inferiority (hard gate, tolerance="
      f"{CUSTOMER_INTELLIGENCE_KPI_TARGETS['composite_non_inferiority']['auc_tolerance']}): "
      f"{CUSTOMER_INTELLIGENCE_KPI_TARGETS['composite_non_inferiority']['hard_gate']}")
print("\n✅ Section 7 complete.")


# =============================================================================
# SECTION 8: WRITE CUSTOMER INTELLIGENCE POLICY ARTIFACT
# =============================================================================
_section("SECTION 8: Write Customer Intelligence Policy Artifact")

CUSTOMER_INTELLIGENCE_POLICY = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "problem": "Problem 12 -- 360 Degree Customer Intelligence (Multi-Signal Customer Profile Aggregation)",
    "unified_profile_fields": UNIFIED_PROFILE_FIELDS,
    "unified_score_weights": UNIFIED_SCORE_WEIGHTS,
    "unified_risk_grade_names": UNIFIED_RISK_GRADE_NAMES,
    "unified_risk_grade_cut_percentiles": UNIFIED_RISK_GRADE_CUT_PERCENTILES,
    "dynamic_pd_eligibility_coverage_pct": DYNAMIC_PD_ELIGIBILITY_COVERAGE_PCT,
    "collections_eligible_coverage_pct_upper_bound": P9_CURE_LABEL_COVERAGE_PCT,
    "reused_from_problem_1": {"champion_model": CHAMPION_NAME, "champion_holdout_auc": STATIC_PD_AUC},
    "reused_from_problem_6": {
        "model_path": str(P6_MODEL_PATH), "preprocessing_path": str(P6_PREPROCESSING_PATH),
        "winning_w": P6_WINNING_W, "recommended_for_production": P6_RECOMMENDED_FOR_PRODUCTION,
    },
    "reused_from_problem_9": {
        "model_path": str(P9_MODEL_PATH), "deployment_policy_path": str(P9_DEPLOYMENT_POLICY_PATH),
        "collections_eligible_states": P9_COLLECTIONS_ELIGIBLE_STATES,
        "treatment_tiers": [t.get("name", t) if isinstance(t, dict) else t for t in P9_TREATMENT_TIERS],
        "recommended_for_production": P9_RECOMMENDED_FOR_PRODUCTION,
    },
    "reused_from_problem_10": {
        "worklist_path": str(P10_WORKLIST_PATH), "deployment_policy_path": str(P10_DEPLOYMENT_POLICY_PATH),
        "risk_level_names": P10_RISK_LEVEL_NAMES, "trend_names": P10_TREND_NAMES,
        "recommended_for_production": P10_RECOMMENDED_FOR_PRODUCTION,
    },
    "kpi_targets": CUSTOMER_INTELLIGENCE_KPI_TARGETS,
    "random_seed": RANDOM_SEED,
    "warp_resource_cap": {
        "cpu_fraction_cap": _PHASE4_CPU_FRACTION_CAP, "ram_fraction_cap": _PHASE4_RAM_FRACTION_CAP,
        "warp_thread_count": WARP_THREAD_COUNT, "max_ram_bytes": MAX_RAM_BYTES,
        "note": "Phase 4 cap (92%/92%) carried forward into Phase 5 -- no new incident to warrant a change.",
    },
}
policy_path = CUSTOMER_INTELLIGENCE_POLICY_DIR / "customer_intelligence_policy.json"
with open(policy_path, "w", encoding="utf-8") as f:
    json.dump(CUSTOMER_INTELLIGENCE_POLICY, f, indent=2)
print(f"Wrote: {policy_path}")
print("\n✅ Section 8 complete.")


# =============================================================================
# SECTION 9: VERIFICATION -- INTEGRITY CHECKS
# =============================================================================
_section("SECTION 9: Verification -- Integrity Checks")


def _check(label, condition, detail=""):
    status = "PASS" if condition else "FAIL"
    print(f"  [{status}] {label}" + (f" -- {detail}" if detail and not condition else ""))
    return condition


_all_checks_passed = True
_all_checks_passed &= _check("Policy file was written", policy_path.exists())
_all_checks_passed &= _check("UNIFIED_RISK_GRADE_NAMES has exactly 3 tiers (tertile convention)",
                              len(UNIFIED_RISK_GRADE_NAMES) == 3)
_all_checks_passed &= _check("Composite weights (static+dynamic) sum to 1.0",
                              abs(UNIFIED_SCORE_WEIGHTS["static_pd_weight"]
                                  + UNIFIED_SCORE_WEIGHTS["dynamic_pd_weight"] - 1.0) < 1e-9)
_all_checks_passed &= _check("Reused Problem 1's real champion AUC (not fabricated)",
                              STATIC_PD_AUC == CHAMPION_METRICS.get("holdout_auc"))
_all_checks_passed &= _check("Reused Problem 6's real winning window verbatim",
                              P6_WINNING_W == NB40_SUMMARY["winning_w"])
_all_checks_passed &= _check("Reused Problem 9's real collections-eligible states verbatim",
                              P9_COLLECTIONS_ELIGIBLE_STATES == P9_DEPLOYMENT_POLICY["collections_eligible_states"])
_all_checks_passed &= _check("Reused Problem 10's real worklist path verbatim",
                              str(P10_WORKLIST_PATH) == NB55_SUMMARY["worklist_path"])
_all_checks_passed &= _check("Both hard-gating KPIs are marked hard_gate=True (not silently advisory)",
                              CUSTOMER_INTELLIGENCE_KPI_TARGETS["profile_completeness"]["hard_gate"] is True
                              and CUSTOMER_INTELLIGENCE_KPI_TARGETS["composite_non_inferiority"]["hard_gate"] is True)
_all_checks_passed &= _check("Dynamic-PD eligibility coverage is a real measured percentage in (0, 100]",
                              0 < DYNAMIC_PD_ELIGIBILITY_COVERAGE_PCT <= 100)
_all_checks_passed &= _check("Unified profile field list has no duplicates",
                              len(UNIFIED_PROFILE_FIELDS) == len(set(UNIFIED_PROFILE_FIELDS)))
_all_checks_passed &= _check("WARP thread count never exceeds the historical config's own value",
                              WARP_THREAD_COUNT <= _historical_thread_count)
_all_checks_passed &= _check("WARP RAM ceiling never exceeds the historical config's own value",
                              MAX_RAM_BYTES <= _historical_max_ram_bytes)

if not _all_checks_passed:
    raise AssertionError("One or more verification checks failed -- see FAIL lines above.")
print("\n✅ Section 9 complete -- all checks passed.")


# =============================================================================
# SECTION 10: WRITE NOTEBOOK 62 SUMMARY ARTIFACT & COMPLETION
# =============================================================================
_section("SECTION 10: Write Notebook 62 Summary Artifact")

NB62_SUMMARY = {
    "notebook": "62_customer_intelligence_business_understanding.ipynb",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "policy_path": str(policy_path),
    "unified_risk_grade_names": UNIFIED_RISK_GRADE_NAMES,
    "unified_score_weights": UNIFIED_SCORE_WEIGHTS,
    "dynamic_pd_eligibility_coverage_pct": DYNAMIC_PD_ELIGIBILITY_COVERAGE_PCT,
    "collections_eligible_coverage_pct_upper_bound": P9_CURE_LABEL_COVERAGE_PCT,
    "warp_thread_count": WARP_THREAD_COUNT,
    "max_ram_bytes": MAX_RAM_BYTES,
    "random_seed": RANDOM_SEED,
}
NB62_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_62_summary.json"
with open(NB62_SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(NB62_SUMMARY, f, indent=2)
print(f"Wrote: {NB62_SUMMARY_PATH}")

_section("NOTEBOOK 62 COMPLETE")
print(f"UNIFIED_RISK_GRADE_NAMES (ASSUMPTION, tertile convention)         : {UNIFIED_RISK_GRADE_NAMES}")
print(f"UNIFIED_SCORE_WEIGHTS (ASSUMPTION)                                : {UNIFIED_SCORE_WEIGHTS['static_pd_weight']} "
      f"static / {UNIFIED_SCORE_WEIGHTS['dynamic_pd_weight']} dynamic / "
      f"{UNIFIED_SCORE_WEIGHTS['collections_adjustment_weight']} collections-adj")
print(f"Dynamic-PD eligibility coverage (real)                            : "
      f"{DYNAMIC_PD_ELIGIBILITY_COVERAGE_PCT:.1f}%")
print("Hard-gating KPIs                                                  : profile_completeness, "
      "composite_non_inferiority")
print(f"WARP cap this notebook forward                                    : "
      f"{WARP_THREAD_COUNT} threads / {MAX_RAM_BYTES / 1e9:.1f} GB RAM")
print(f"Policy written to: {policy_path}")
print(
    "\nNext: 63_customer_intelligence_modeling.ipynb -- scores every eligible real customer with Problem 1's "
    "real champion model (static PD), Problem 6's real persisted model (dynamic PD), joins Problem 9's real "
    "propensity-to-cure score where collections-eligible, joins Problem 10's real scored worklist "
    "(risk-level/trend/action), computes the real UNIFIED_RISK_SCORE per Section 6's composite formula, fits "
    "the real tertile cut values on that population, and validates both hard-gating KPIs set here against the "
    "real observed default outcome."
)
